<a href="https://colab.research.google.com/github/Deodael/freecodecamp-data-analysis/blob/main/sms_text_classification_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import pandas as pd
import numpy as np
import urllib.request

# 1. Download the data files safely using a browser User-Agent header
TRAIN_URL = "https://cdn.freecodecamp.org/project-data/sms/train-data.tsv"
VALID_URL = "https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv"

req_train = urllib.request.Request(TRAIN_URL, headers={'User-Agent': 'Mozilla/5.0'})
req_valid = urllib.request.Request(VALID_URL, headers={'User-Agent': 'Mozilla/5.0'})

# 2. Load directly into Pandas DataFrames
df_train_local = pd.read_csv(urllib.request.urlopen(req_train), sep='\t', header=None, names=['type', 'text'])
df_test_local = pd.read_csv(urllib.request.urlopen(req_valid), sep='\t', header=None, names=['type', 'text'])

# 3. Map text labels ('ham'/'spam') into numbers (0/1)
df_train_local['label'] = df_train_local['type'].map({'ham': 0, 'spam': 1})
df_test_local['label'] = df_test_local['type'].map({'ham': 0, 'spam': 1})

# FIX: Force the numpy arrays to a generic Python object dtype
train_text = df_train_local['text'].values.astype('O')
train_labels = df_train_local['label'].values.astype(np.float32)
test_text = df_test_local['text'].values.astype('O')
test_labels = df_test_local['label'].values.astype(np.float32)

# 4. Build the text vectorizer layer
VOCAB_SIZE = 10000
MAX_LEN = 250

vectorizer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)
vectorizer.adapt(train_text)

# 5. Assemble the Embedding Sequential model
model = tf.keras.Sequential([
    layers.Input(shape=(1,), dtype=tf.string),
    vectorizer,
    layers.Embedding(VOCAB_SIZE, 64),
    layers.GlobalAveragePooling1D(),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

# 6. Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 7. Train the model for 10 epochs
model.fit(
    train_text,
    train_labels,
    epochs=10,
    batch_size=32,
    validation_data=(test_text, test_labels),
    verbose=1
)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
def predict_message(pred_text):
    # 1. Wrap the incoming sentence into a numpy object array so the model can read it
    text_array = np.array([pred_text], dtype='O')

    # 2. Extract the decimal score from the model's prediction matrix
    probability = float(model.predict(text_array)[0][0])

    # 3. Assign the correct category label based on the 0.5 classification threshold
    label = "spam" if probability >= 0.5 else "ham"

    # 4. Return the exact format freeCodeCamp's test cell checks for
    return [probability, label]

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
